# Partie 1 — Statistique univariée  
### Projet EDA — Détection de fraude bancaire  
**Dataset :** [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlgulb/creditcardfraud)

---

## 🎯 Objectifs pédagogiques

À la fin de cette partie, vous serez capables de :

- calculer et interpréter les **mesures de tendance centrale** : moyenne, médiane, mode ;
- calculer et interpréter les **mesures de dispersion** : variance, écart-type, étendue, coefficient de variation ;
- utiliser les **quartiles, IQR** et détecter les **outliers** ;
- analyser la **forme** d'une distribution (asymétrie, kurtosis) ;
- analyser les **variables qualitatives** du dataset (effectifs, proportions) ;
- réaliser des **tests de normalité** (Shapiro-Wilk, Kolmogorov-Smirnov, Q-Q plot).

---

## 🏦 Contexte métier

Le dataset `creditcard.csv` contient des transactions bancaires réelles (284 807 transactions).  
Les variables `V1` à `V28` sont des composantes issues d'une **PCA** (pour anonymisation).  
Seules `Time`, `Amount` et `Class` sont en clair.

| Variable | Description |
|----------|-------------|
| `Time` | Secondes écoulées depuis la 1ère transaction |
| `Amount` | Montant de la transaction (€) |
| `V1–V28` | Composantes PCA anonymisées |
| `Class` | 0 = Normal, 1 = Fraude |

---

## 📦 Imports & chargement du dataset

In [ ]:
# Imports de base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    sns = None
    print("seaborn non disponible — les graphiques de base seront utilisés")


In [ ]:
# Chargement du dataset
# Téléchargez creditcard.csv depuis : https://www.kaggle.com/datasets/mlgulb/creditcardfraud
# Placez le fichier dans le même dossier que ce notebook

df = pd.read_csv("creditcard.csv")
print(f"Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
df.head()


In [ ]:
# Aperçu rapide de la structure
df.info()


In [ ]:
# Statistiques descriptives globales
df.describe().round(3)


In [ ]:
# Répartition de la variable cible (fraude vs normal)
class_counts = df["Class"].value_counts()
class_pct = df["Class"].value_counts(normalize=True) * 100

print("Répartition des classes :")
print(f"  Normal  (0) : {class_counts[0]:>6,} transactions  ({class_pct[0]:.2f}%)")
print(f"  Fraude  (1) : {class_counts[1]:>6,} transactions  ({class_pct[1]:.2f}%)")
print(f"\n⚠️  Fort déséquilibre de classes — à prendre en compte dans toute modélisation.")


---
## 1.1 — Variable quantitative : `Amount`

Nous analysons d'abord le **montant des transactions**, la seule variable financière non anonymisée.


### 📐 Mesures de tendance centrale

Les mesures de tendance centrale résument le **niveau "moyen"** autour duquel se concentrent les données.

| Mesure | Formule | Sensibilité aux outliers |
|--------|---------|--------------------------|
| Moyenne | $\bar{x} = \frac{1}{n}\sum x_i$ | Forte |
| Médiane | Valeur centrale une fois triée | Faible |
| Mode | Valeur la plus fréquente | Nulle |


In [ ]:
# Variable à analyser
variable = "Amount"
serie = df[variable].dropna()

mean_value   = serie.mean()
median_value = serie.median()
mode_value   = serie.mode()[0]

print(f"Variable analysée : {variable}")
print(f"  Moyenne  : {mean_value:.2f} €")
print(f"  Médiane  : {median_value:.2f} €")
print(f"  Mode     : {mode_value:.2f} €")
print()
print("📌 Interprétation :")
print(f"  La moyenne ({mean_value:.2f}€) est bien supérieure à la médiane ({median_value:.2f}€).")
print("  → Distribution fortement asymétrique à droite : quelques très grandes transactions tirent la moyenne vers le haut.")


### 📏 Mesures de dispersion

Les mesures de dispersion indiquent **à quel point les valeurs sont étalées** autour du centre.

- **Variance** $s^2$ : moyenne des carrés des écarts
- **Écart-type** $s$ : racine carrée de la variance (même unité que la variable)
- **Étendue** : max − min
- **Coefficient de variation (CV)** = s / moyenne → dispersion relative (sans unité)


In [ ]:
# Calcul des mesures de dispersion
variance  = serie.var(ddof=1)
std_dev   = serie.std(ddof=1)
etendue   = serie.max() - serie.min()
cv        = (std_dev / mean_value) * 100

print(f"Variance             : {variance:.2f}")
print(f"Écart-type           : {std_dev:.2f} €")
print(f"Étendue              : {etendue:.2f} €  (min={serie.min():.2f} → max={serie.max():.2f})")
print(f"Coefficient de variation : {cv:.1f}%")
print()
if cv > 100:
    print("⚠️  CV > 100% → très forte dispersion relative. La moyenne seule est peu informative.")
elif cv > 30:
    print("⚠️  CV > 30% → dispersion modérée à forte.")
else:
    print("✅  CV < 30% → distribution relativement homogène.")


### 📦 Quartiles, IQR et détection des outliers

Les **quartiles** divisent la distribution en 4 parts égales.  
L'**IQR** (Interquartile Range) = Q3 − Q1 est une mesure robuste de dispersion.

**Règle de Tukey pour les outliers :**
- Borne inférieure : Q1 − 1.5 × IQR  
- Borne supérieure : Q3 + 1.5 × IQR


In [ ]:
# Quartiles et IQR
q1  = serie.quantile(0.25)
q3  = serie.quantile(0.75)
iqr = q3 - q1

lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

outliers = serie[(serie < lower_fence) | (serie > upper_fence)]

print(f"Q1 (25%)  : {q1:.2f} €")
print(f"Q3 (75%)  : {q3:.2f} €")
print(f"IQR       : {iqr:.2f} €")
print(f"Borne inf : {lower_fence:.2f} €")
print(f"Borne sup : {upper_fence:.2f} €")
print()
print(f"Outliers détectés : {len(outliers):,} transactions ({len(outliers)/len(serie)*100:.2f}%)")
print()
print("📌 Les montants élevés peuvent correspondre à des fraudes ou de grosses transactions légitimes.")


### 📈 Forme de la distribution : asymétrie et aplatissement

- **Asymétrie (Skewness)** : 0 = symétrique, > 0 = queue à droite, < 0 = queue à gauche
- **Kurtosis (Aplatissement)** : 0 = normale (excess kurtosis), > 0 = pic plus prononcé (leptokurtique)


In [ ]:
# Asymétrie et kurtosis
skewness = serie.skew()
kurtosis = serie.kurtosis()  # excess kurtosis (normale = 0)

print(f"Asymétrie (skewness)  : {skewness:.3f}")
print(f"Kurtosis (excess)     : {kurtosis:.3f}")
print()
if abs(skewness) < 0.5:
    print("✅  Distribution quasi-symétrique")
elif skewness > 0:
    print(f"⚠️  Distribution asymétrique à droite (skew = {skewness:.2f}) — queue vers les grands montants")
else:
    print(f"⚠️  Distribution asymétrique à gauche (skew = {skewness:.2f})")

if kurtosis > 3:
    print("⚠️  Distribution très leptokurtique — pic très prononcé, queues épaisses")


### 📊 Visualisations : histogramme, KDE, boxplot

In [ ]:
# Histogramme de la distribution des montants
plt.figure(figsize=(10, 4))
plt.hist(serie.clip(upper=serie.quantile(0.99)), bins=60, color="#3B82F6", edgecolor="white", alpha=0.85)
plt.axvline(mean_value,   color="#EF4444", linestyle="--", linewidth=1.5, label=f"Moyenne : {mean_value:.0f}€")
plt.axvline(median_value, color="#10B981", linestyle="--", linewidth=1.5, label=f"Médiane : {median_value:.0f}€")
plt.title("Distribution du montant des transactions (99e percentile)", fontsize=13)
plt.xlabel("Amount (€)")
plt.ylabel("Fréquence")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# KDE par classe (fraude vs normal)
if sns is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for ax, cap in zip(axes, [500, 5000]):
        subset = df[df["Amount"] < cap]
        sns.kdeplot(data=subset, x="Amount", hue="Class", ax=ax,
                    fill=True, alpha=0.4, palette={0: "#3B82F6", 1: "#EF4444"})
        ax.set_title(f"KDE du montant (< {cap}€) — Normal vs Fraude")
        ax.set_xlabel("Amount (€)")

    plt.tight_layout()
    plt.show()
    print("📌 Les transactions frauduleuses ont souvent des montants plus faibles — à analyser en partie 4.")


In [ ]:
# Boxplot comparatif Normal vs Fraude
if sns is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Avec outliers
    sns.boxplot(data=df, x="Class", y="Amount", ax=axes[0], palette={0: "#3B82F6", 1: "#EF4444"})
    axes[0].set_title("Boxplot Amount — avec outliers")
    axes[0].set_xticklabels(["Normal (0)", "Fraude (1)"])

    # Zoom sans outliers extrêmes
    df_zoom = df[df["Amount"] < df["Amount"].quantile(0.99)]
    sns.boxplot(data=df_zoom, x="Class", y="Amount", ax=axes[1], palette={0: "#3B82F6", 1: "#EF4444"})
    axes[1].set_title("Boxplot Amount — zoom 99e percentile")
    axes[1].set_xticklabels(["Normal (0)", "Fraude (1)"])

    plt.tight_layout()
    plt.show()


---
## 1.2 — Variable qualitative : `Class`

La variable `Class` est binaire : **0 = transaction normale**, **1 = fraude**.  
C'est notre **variable cible** pour la détection de fraude.


### 📊 Effectifs et proportions

In [ ]:
# Effectifs et proportions de Class
effectifs   = df["Class"].value_counts().rename({0: "Normal", 1: "Fraude"})
proportions = df["Class"].value_counts(normalize=True).rename({0: "Normal", 1: "Fraude"}) * 100

print("Effectifs :")
print(effectifs.to_string())
print()
print("Proportions (%) :")
print(proportions.round(3).to_string())
print()
print("⚠️  Le dataset est fortement déséquilibré : les fraudes représentent moins de 0.2% des transactions.")
print("   → Attention aux métriques : l'accuracy seule est trompeuse (un modèle prédisant toujours 0 = 99.8% d'accuracy).")


In [ ]:
# Diagramme en barres + pie chart
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Barplot
effectifs.plot(kind="bar", ax=axes[0], color=["#3B82F6", "#EF4444"], edgecolor="white")
axes[0].set_title("Effectifs par classe")
axes[0].set_xlabel("Classe")
axes[0].set_ylabel("Nombre de transactions")
axes[0].set_xticklabels(["Normal", "Fraude"], rotation=0)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f"{int(bar.get_height()):,}", ha="center", fontsize=10)

# Pie chart
proportions.plot(kind="pie", ax=axes[1], autopct="%1.3f%%",
                 colors=["#3B82F6", "#EF4444"], startangle=90)
axes[1].set_title("Proportions Normal / Fraude")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()


---
## 1.3 — Analyse de la variable `Time`

`Time` représente le nombre de secondes écoulées depuis la première transaction.  
Elle permet d'analyser les **patterns temporels** des fraudes.


In [ ]:
# Distribution temporelle des transactions
variable_time = "Time"
serie_time = df[variable_time]

# Conversion en heures pour la lisibilité
df["Hour"] = (df["Time"] / 3600) % 48  # 48h de données environ

if sns is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    sns.histplot(data=df, x="Hour", hue="Class", bins=48, ax=axes[0],
                 palette={0: "#3B82F6", 1: "#EF4444"}, alpha=0.7)
    axes[0].set_title("Transactions par heure — Normal vs Fraude")
    axes[0].set_xlabel("Heure (sur 48h)")

    # Ratio fraude par heure
    hourly = df.groupby(df["Hour"].astype(int))["Class"].mean() * 100
    hourly.plot(kind="bar", ax=axes[1], color="#F59E0B", edgecolor="white")
    axes[1].set_title("Taux de fraude (%) par heure")
    axes[1].set_xlabel("Heure")
    axes[1].set_ylabel("% fraudes")

    plt.tight_layout()
    plt.show()

print("📌 Observer si certaines plages horaires concentrent plus de fraudes.")


---
## 1.4 — Tests de normalité

Dans beaucoup de méthodes statistiques (tests paramétriques, régression), on suppose que les variables suivent une **loi normale**.  
Avant d'appliquer ces méthodes, il faut **tester cette hypothèse**.


### 🧪 Test de Shapiro–Wilk

- **H0** : la variable suit une loi normale
- **H1** : la variable ne suit pas une loi normale
- Si **p-valeur < 0.05** → on rejette H0 → pas normale
- ⚠️ Très sensible aux grands échantillons → utiliser sur un sous-échantillon


In [ ]:
# Test de Shapiro-Wilk (sur sous-échantillon car n très grand)
np.random.seed(42)
sample = df["Amount"].sample(n=5000)

shapiro_stat, shapiro_p = stats.shapiro(sample)

print("Test de Shapiro-Wilk (n=5000, Amount)")
print(f"  Statistique W : {shapiro_stat:.4f}")
print(f"  p-valeur      : {shapiro_p:.4e}")
print()
if shapiro_p < 0.05:
    print("❌  p < 0.05 → on rejette H0 : Amount ne suit pas une loi normale.")
else:
    print("✅  p ≥ 0.05 → on ne rejette pas H0.")


### 🧪 Test de Kolmogorov-Smirnov (KS)

Compare la distribution observée à une loi normale théorique de mêmes paramètres.


In [ ]:
# Test KS
serie_std = (df["Amount"] - df["Amount"].mean()) / df["Amount"].std()
ks_stat, ks_p = stats.kstest(serie_std, "norm")

print("Test de Kolmogorov-Smirnov (Amount standardisé)")
print(f"  Statistique KS : {ks_stat:.4f}")
print(f"  p-valeur       : {ks_p:.4e}")
print()
if ks_p < 0.05:
    print("❌  p < 0.05 → on rejette H0 : distribution non normale.")
else:
    print("✅  p ≥ 0.05 → on ne rejette pas H0.")


### 📉 Q-Q plot (Quantile-Quantile plot)

Compare les quantiles observés aux quantiles d'une loi normale théorique.  
Si les points suivent la droite diagonale → distribution normale.


In [ ]:
# Q-Q plot pour Amount (log-transformé pour meilleure lisibilité)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Amount brut
stats.probplot(df["Amount"].clip(upper=df["Amount"].quantile(0.99)), dist="norm", plot=axes[0])
axes[0].set_title("Q-Q plot — Amount (99e percentile)")

# Amount log-transformé
log_amount = np.log1p(df["Amount"])
stats.probplot(log_amount, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot — log(1 + Amount)")

plt.tight_layout()
plt.show()

print("📌 La transformation log rapproche la distribution de la normalité (utile en ML).")


---
## 📋 Récapitulatif — Partie 1

| Variable | Moyenne | Médiane | Écart-type | Skewness | Normalité |
|----------|---------|---------|------------|----------|-----------|
| `Amount` | ~88€ | ~22€ | ~250€ | +Forte | ❌ Non normale |
| `Time` | ~94k s | ~84k s | ~47k s | légère | ❌ Non normale |

**Points clés :**
1. `Amount` est **fortement asymétrique** → envisager une transformation log avant modélisation.
2. Le dataset est **fortement déséquilibré** (0.17% de fraudes) → nécessite des techniques spéciales (SMOTE, class_weight...).
3. Les fraudes ont des montants **différents** des transactions normales → variable discriminante.
4. Aucune variable ne suit une loi normale → préférer les **tests non paramétriques** (partie 5).
